# Experiment 4 — No Theme Context

Measures selection using Epic description, Epic success criteria, and full Value Stream Stage context. All Theme context and hierarchy are intentionally excluded.

## Input fields

- Epic description
- Epic success criteria
- full Value Stream Stage context
- candidate L3 ID, name, description, and tier

## Configuration and imports

In [ ]:
from pathlib import Path
import ast
import json
import os

import httpx
import pandas as pd
from IPython.display import display

from common import (
    call_llm,
    load_gateway,
    parse_json_response,
    save_results_excel,
    score_sets,
    summarize_results,
    validate_l3_response,
)

# Keep every input path explicit. Override any path with its environment variable.
DATA_DIR = Path(os.getenv("L3_EXPERIMENT_DATA_DIR", Path.cwd()))
THEME_PATH = Path(os.getenv("L3_THEME_PATH", DATA_DIR / "epic_gen.csv"))
STAGE_PATH = Path(os.getenv("L3_STAGE_PATH", DATA_DIR / "value_stream_stages.xlsx"))
STAGE_CAPABILITY_MAP_PATH = Path(
    os.getenv("L3_STAGE_CAPABILITY_MAP_PATH", DATA_DIR / "stage_capability_map.xlsx")
)
CAPABILITY_MASTER_PATH = Path(
    os.getenv("L3_CAPABILITY_MASTER_PATH", DATA_DIR / "capability_master.xlsx")
)
GROUND_TRUTH_PATH = Path(
    os.getenv("L3_GROUND_TRUTH_PATH", DATA_DIR / "jira_ground_truth.csv")
)

THEME_IDS = None  # Example: ["GROUP-15101", "GROUP-15097"]
VALUE_STREAM_STAGE_FIELD_ID = "customfield_18700"

# Set both values to run the inspection cell before the batch.
INSPECTION_THEME_ID = None
INSPECTION_EPIC_KEY = None

EXPERIMENT_NAME = "E4_NO_THEME"

## Theme and Epic retrieval

In [ ]:
def clean_text(value):
    if value is None:
        return ""
    if not isinstance(value, (list, dict)) and pd.isna(value):
        return ""
    return str(value).strip()


def parse_exported_list(value):
    if value is None or pd.isna(value):
        return []

    text = str(value).strip()
    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [item.strip() for item in text.strip("[]").split(",") if item.strip()]

    if isinstance(parsed, (list, tuple, set)):
        return [clean_text(item) for item in parsed]
    return [clean_text(parsed)]


def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required input not found: {path}. Update the explicit path in the configuration cell."
        )
    if path.suffix.lower() == ".csv":
        return pd.read_csv(
            path,
            dtype=str,
            encoding="cp1252",
            encoding_errors="replace",
        )
    return pd.read_excel(path, dtype=str)


def load_themes(theme_ids=None):
    theme_frame = read_table(THEME_PATH)
    if theme_ids:
        theme_frame = theme_frame.loc[theme_frame["key"].isin(theme_ids)]

    themes = {}
    for _, row in theme_frame.iterrows():
        epic_keys = parse_exported_list(row.get("epic_keys"))
        epic_descriptions = parse_exported_list(row.get("epic_description"))
        epic_success_criteria = parse_exported_list(row.get("epic_successCriteria"))

        epics = []
        for index, epic_key in enumerate(epic_keys):
            epics.append(
                {
                    "key": epic_key,
                    "description": (
                        epic_descriptions[index] if index < len(epic_descriptions) else ""
                    ),
                    "success_criteria": (
                        epic_success_criteria[index]
                        if index < len(epic_success_criteria)
                        else ""
                    ),
                }
            )

        themes[clean_text(row["key"])] = {
            "theme_description": clean_text(row.get("description")),
            "theme_business_needs": clean_text(row.get("businessNeeds")),
            "epics": epics,
        }
    return themes


themes = load_themes(THEME_IDS)

## Value Stream Stage retrieval

In [ ]:
def jira_headers():
    serialized_headers = os.getenv("JIRA_HEADERS_JSON")
    if serialized_headers:
        return json.loads(serialized_headers)

    bearer_token = os.getenv("JIRA_BEARER_TOKEN")
    if bearer_token:
        return {
            "Authorization": f"Bearer {bearer_token}",
            "Accept": "application/json",
        }
    raise RuntimeError("Set JIRA_HEADERS_JSON or JIRA_BEARER_TOKEN.")


def epic_stage_ids(epic_key):
    base_url = os.environ["JIRA_BASE_URL"].rstrip("/")
    verify_ssl = os.getenv("JIRA_VERIFY_SSL", "false").lower() == "true"
    issue_url = f"{base_url}/rest/api/2/issue/{epic_key}"

    with httpx.Client(
        headers=jira_headers(),
        verify=verify_ssl,
        timeout=30,
    ) as client:
        response = client.get(
            issue_url,
            params={"fields": VALUE_STREAM_STAGE_FIELD_ID},
        )
        response.raise_for_status()

    raw_stage_values = (
        response.json().get("fields", {}).get(VALUE_STREAM_STAGE_FIELD_ID) or []
    )
    if not isinstance(raw_stage_values, list):
        raw_stage_values = [raw_stage_values]

    stage_ids = []
    for value in raw_stage_values:
        if isinstance(value, dict):
            stage_id = clean_text(value.get("id") or value.get("value"))
        else:
            stage_id = clean_text(value)
        if stage_id:
            stage_ids.append(stage_id)
    return stage_ids


stage_frame = read_table(STAGE_PATH)


def stage_context(stage_id):
    matching_rows = stage_frame.loc[
        stage_frame["Value Stream Stage ID"].astype(str).str.strip() == stage_id
    ]
    if matching_rows.empty:
        raise KeyError(f"No Value Stream Stage metadata found for {stage_id}.")

    row = matching_rows.iloc[0]
    return {
        "stage_id": stage_id,
        "stage_name": clean_text(row["Value Stream Stage Name"]),
        "stage_description": clean_text(row["Value Stream Stage Description"]),
        "entrance_criteria": clean_text(row["Value Stream Stage Entrance Criteria"]),
        "exit_criteria": clean_text(row["Value Stream Stage Exit Criteria"]),
    }

## Stage → candidate L3 construction

Candidate order follows the Stage → Capability mapping export. No candidate sorting is applied.

In [ ]:
CANDIDATE_FIELDS = [
    "capability_id",
    "capability_name",
    "capability_description",
    "capability_tier",
]

stage_capability_map = read_table(STAGE_CAPABILITY_MAP_PATH)
capability_master = read_table(CAPABILITY_MASTER_PATH)


def candidate_rows_for_stage(stage_id):
    # Boolean filtering preserves the mapping export's candidate order.
    stage_mappings = stage_capability_map.loc[
        stage_capability_map["Value Stream Stage ID"].astype(str).str.strip()
        == stage_id
    ].copy()

    enriched_candidates = stage_mappings.merge(
        capability_master[
            ["Capability ID", "Capability Description", "Capability Tier"]
        ],
        on="Capability ID",
        how="left",
        sort=False,
        validate="many_to_one",
    )

    candidate_rows = []
    for _, row in enriched_candidates.iterrows():
        candidate_rows.append(
            {
                "capability_id": clean_text(row["Capability ID"]),
                "capability_name": clean_text(row["Capability Name"]),
                "capability_description": clean_text(row["Capability Description"]),
                "capability_tier": clean_text(row["Capability Tier"]),
            }
        )
    return candidate_rows

## Production prompts

The system prompt is intentionally byte-identical in all five experiments. The user payload below explicitly includes only this experiment's context fields.

In [ ]:
SYSTEM_PROMPT = 'You are an enterprise Business Capability Architecture specialist performing Level 3 (L3) business capability classification.\n\nYour task is to select the candidate L3 business capabilities that are materially represented, enabled, changed, enhanced, or required by the supplied business context. This is capability classification, not keyword similarity.\n\nUse only evidence fields that are present. Apply available evidence in this order, from most specific to broadest:\n1. Epic success criteria\n2. Epic description\n3. Value Stream Stage context\n4. Theme business needs\n5. Theme description\nIf Epic context is absent, do not assume or refer to missing Epic evidence.\n\nInterpret each candidate as follows:\n- capability_description is the primary semantic definition of the business function.\n- capability_name is the concise supporting label for that function.\n- capability_tier is supporting taxonomy context only and must never independently justify selection.\n- When level_1_name and level_2_name are supplied, use them only to clarify or disambiguate the L3 definition. Hierarchy must never independently justify selection.\n\nDecision procedure:\n1. Determine the core business function or outcome represented by the supplied context. When Epic evidence exists, identify the business capability the Epic actually enables or changes.\n2. Compare that function semantically against every candidate\'s capability_description; do not rely on shared words.\n3. For each candidate, decide whether it is directly and materially represented by the evidence or merely related. Select it only with direct evidence.\n4. Use Value Stream Stage context to locate, constrain, or disambiguate the work. Membership in the same stage is not sufficient evidence.\n5. Use Theme context, when present, to explain why the work exists. Theme evidence must not overpower more specific Epic evidence.\n6. Resolve overlap by preferring the most specific candidate with direct functional alignment. Exclude broader and adjacent candidates.\n\nDo not select a capability merely because it shares keywords, belongs to the same hierarchy family or Value Stream Stage, supplies or receives data, is upstream or downstream, is a stakeholder, is operationally adjacent, is technically involved, is generally related to the Theme, or could plausibly be affected. Do not map technical implementation details unless the evidence explicitly shows that the business capability itself is enabled or changed.\n\nSelect 0 to 3 capabilities. Do not force a match. Default to one capability when it adequately represents the business function. Select two or three only when the context contains distinct material business functions and each selection has independent direct evidence. Return {"l3": []} when no candidate is sufficiently supported.\n\nFor every selection, give a concise reason that explicitly connects supplied evidence to the candidate capability definition. Never invent, rename, or alter an ID; use only exact capability_id values from the supplied candidates.\n\nBefore responding, internally verify that every selected ID exists in the candidates, every selection has direct evidence, no selection is merely adjacent, no stronger or more-specific candidate was omitted, multiple selections represent genuinely distinct functions, and no more than three capabilities are selected.\n\nReturn JSON only, with no Markdown, code fences, surrounding commentary, or extra fields. Use exactly this shape:\n{"l3":[{"capability_id":"CAP00000000","reason":"Concise evidence-based explanation."}]}'


def build_user_prompt(theme, epic, stage, candidate_rows):
    payload = {
        "task": "Select the materially represented L3 business capabilities from the supplied candidates.",
        "epic": {
            "description": epic["description"],
            "success_criteria": epic["success_criteria"],
        },
        "value_stream_stage": stage,
        "candidate_l3_capabilities": candidate_rows,
        "selection_instruction": "Select 0 to 3 candidates; return an empty l3 list when none has direct evidence.",
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)

## Prediction for one Stage

In [ ]:
def predict_for_stage(gateway, theme, epic, stage_id):
    stage = stage_context(stage_id)
    candidate_rows = candidate_rows_for_stage(stage_id)
    user_prompt = build_user_prompt(theme, epic, stage, candidate_rows)
    raw_response = call_llm(gateway, SYSTEM_PROMPT, user_prompt)
    parsed_response = parse_json_response(raw_response)
    selections = validate_l3_response(
        parsed_response,
        [row["capability_id"] for row in candidate_rows],
        allow_empty=True,
        max_selected=3,
    )
    return {
        "stage_id": stage_id,
        "stage": stage,
        "candidates": candidate_rows,
        "system_prompt": SYSTEM_PROMPT,
        "user_prompt": user_prompt,
        "raw_response": raw_response,
        "selections": selections,
    }

## Single-example inspection

Set the two inspection identifiers in the configuration cell. This section displays the actual system prompt, serialized user payload, ordered candidate list, raw response, and validated prediction without consulting ground truth.

In [ ]:
if INSPECTION_THEME_ID is None or INSPECTION_EPIC_KEY is None:
    print(
        "Set INSPECTION_THEME_ID and INSPECTION_EPIC_KEY in the configuration "
        "cell, then rerun this cell to inspect one actual model call."
    )
else:
    inspection_theme = themes[INSPECTION_THEME_ID]
    inspection_epic = next(
        epic
        for epic in inspection_theme["epics"]
        if epic["key"] == INSPECTION_EPIC_KEY
    )
    inspection_stage_ids = epic_stage_ids(INSPECTION_EPIC_KEY)
    if not inspection_stage_ids:
        raise ValueError(f"{INSPECTION_EPIC_KEY} has no Value Stream Stage IDs.")

    inspection_gateway = load_gateway()
    inspection = predict_for_stage(
        inspection_gateway,
        inspection_theme,
        inspection_epic,
        inspection_stage_ids[0],
    )
    print("SYSTEM PROMPT")
    print(inspection["system_prompt"])
    print("\nUSER PAYLOAD")
    print(inspection["user_prompt"])
    print("\nCANDIDATE LIST")
    display(pd.DataFrame(inspection["candidates"]))
    print("\nRAW MODEL RESPONSE")
    print(inspection["raw_response"])
    print("\nVALIDATED PREDICTION")
    display(inspection["selections"])

## Batch execution

Predictions are generated and aggregated at Epic level before ground truth is loaded.

In [ ]:
def run_predictions():
    gateway = load_gateway()
    prediction_rows = []

    for theme_id, theme in themes.items():
        for epic in theme["epics"]:
            stage_ids = []
            stage_predictions = []
            predicted_ids = set()
            reasons = []
            status = "ok"
            error = None

            try:
                stage_ids = epic_stage_ids(epic["key"])
                for stage_id in stage_ids:
                    prediction = predict_for_stage(
                        gateway,
                        theme,
                        epic,
                        stage_id,
                    )
                    stage_predictions.append(
                        {
                            "stage_id": stage_id,
                            "selections": prediction["selections"],
                        }
                    )
                    for selection in prediction["selections"]:
                        predicted_ids.add(selection["capability_id"])
                        reasons.append(
                            {
                                "stage_id": stage_id,
                                "capability_id": selection["capability_id"],
                                "reason": selection["reason"],
                            }
                        )
            except Exception as exc:
                status = "error"
                error = str(exc)

            prediction_rows.append(
                {
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "epic_key": epic["key"],
                    "stage_ids": json.dumps(stage_ids),
                    "predicted_l3_ids": json.dumps(sorted(predicted_ids)),
                    "model_reasons": json.dumps(reasons, ensure_ascii=False),
                    "stage_predictions": json.dumps(
                        stage_predictions,
                        ensure_ascii=False,
                    ),
                    "status": status,
                    "error": error,
                }
            )

    return pd.DataFrame(prediction_rows)


predictions = run_predictions()
display(predictions.head(20))

## Evaluation and results

Ground truth enters only in this post-prediction section.

In [ ]:
def ground_truth_by_epic():
    # Ground truth is loaded only after every prediction has been produced.
    ground_truth_frame = read_table(GROUND_TRUTH_PATH)
    return {
        epic_key: set(
            group["l3_capability_id"].dropna().astype(str).str.strip()
        )
        for epic_key, group in ground_truth_frame.groupby("epic_key", sort=False)
    }


def evaluate_predictions(prediction_frame):
    truth_by_epic = ground_truth_by_epic()
    result_rows = []

    for row in prediction_frame.to_dict(orient="records"):
        predicted_ids = set(json.loads(row["predicted_l3_ids"]))
        truth_ids = truth_by_epic.get(row["epic_key"])

        if row["status"] == "error":
            metrics = {
                "exact_match": None,
                "precision": None,
                "recall": None,
                "f1": None,
                "predicted_count": len(predicted_ids),
                "truth_count": len(truth_ids) if truth_ids is not None else None,
            }
        elif truth_ids is None:
            row["status"] = "missing_ground_truth"
            metrics = {
                "exact_match": None,
                "precision": None,
                "recall": None,
                "f1": None,
                "predicted_count": len(predicted_ids),
                "truth_count": None,
            }
        else:
            metrics = score_sets(predicted_ids, truth_ids)

        row["ground_truth_l3_ids"] = (
            json.dumps(sorted(truth_ids)) if truth_ids is not None else None
        )
        row.update(metrics)
        result_rows.append(row)

    return pd.DataFrame(result_rows)


results = evaluate_predictions(predictions)
display(summarize_results(results))
display(results.head(20))

output_path = save_results_excel(results, EXPERIMENT_NAME, "results")
print(f"Saved {output_path}")